In [4]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# -----------------------------
# 1. Dataset (English → Nepali)
# -----------------------------
data = [
    ("hello", "नमस्ते"),
    ("how are you", "तिमीलाई कस्तो छ"),
    ("i am fine", "म ठिक छु"),
    ("what is your name", "तिम्रो नाम के हो"),
    ("i love nepal", "म नेपाललाई माया गर्छु")
]

input_texts = [pair[0] for pair in data]
target_texts = ["<start> " + pair[1] + " <end>" for pair in data]

# -----------------------------
# 2. Tokenization
# -----------------------------
input_tokenizer = Tokenizer()
target_tokenizer = Tokenizer(filters='')

input_tokenizer.fit_on_texts(input_texts)
target_tokenizer.fit_on_texts(target_texts)

input_seq = input_tokenizer.texts_to_sequences(input_texts)
target_seq = target_tokenizer.texts_to_sequences(target_texts)

input_vocab_size = len(input_tokenizer.word_index) + 1
target_vocab_size = len(target_tokenizer.word_index) + 1

max_input_len = max(len(seq) for seq in input_seq)
max_target_len = max(len(seq) for seq in target_seq)

encoder_input_data = pad_sequences(input_seq, maxlen=max_input_len, padding='post')
decoder_input_data = pad_sequences(target_seq, maxlen=max_target_len, padding='post')

# -----------------------------
# 3. Prepare Decoder Target Data
# -----------------------------
decoder_target_data = np.zeros((len(data), max_target_len, target_vocab_size))

for i, seq in enumerate(target_seq):
    for t in range(1, len(seq)):
        decoder_target_data[i, t-1, seq[t]] = 1

# -----------------------------
# 4. Encoder
# -----------------------------
encoder_inputs = Input(shape=(max_input_len,))
enc_emb = Embedding(input_vocab_size, 128)(encoder_inputs)

encoder_lstm = LSTM(256, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)

encoder_states = [state_h, state_c]

# -----------------------------
# 5. Decoder
# -----------------------------
decoder_inputs = Input(shape=(max_target_len,))
dec_emb_layer = Embedding(target_vocab_size, 128)
dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = LSTM(256, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(target_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# -----------------------------
# 6. Seq2Seq Model
# -----------------------------
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(optimizer='adam', loss='categorical_crossentropy')
model.summary()

# -----------------------------
# 7. Train Model
# -----------------------------
model.fit([encoder_input_data, decoder_input_data],
          decoder_target_data,
          epochs=300,
          verbose=1)

# -----------------------------
# 8. Inference Models
# -----------------------------
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference
decoder_state_input_h = Input(shape=(256,))
decoder_state_input_c = Input(shape=(256,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_inputs_single = Input(shape=(1,))
dec_emb2 = dec_emb_layer(decoder_inputs_single)

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2, initial_state=decoder_states_inputs)

decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model(
    [decoder_inputs_single] + decoder_states_inputs,
    [decoder_outputs2, state_h2, state_c2]
)

# -----------------------------
# 9. Reverse Mapping
# -----------------------------
reverse_target_word_index = {i: w for w, i in target_tokenizer.word_index.items()}

# -----------------------------
# 10. Decode Function
# -----------------------------
def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq)

    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_tokenizer.word_index['<start>']

    decoded_sentence = ""

    for _ in range(max_target_len):
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_word_index.get(sampled_token_index, "")

        if sampled_word == "<end>":
            break

        decoded_sentence += " " + sampled_word

        target_seq[0, 0] = sampled_token_index
        states_value = [h, c]

    return decoded_sentence.strip()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_9       │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_10      │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, 4, 128)    │      1,792 │ input_layer_9[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, 6, 128)    │      2,176 │ input_layer_10[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ [(None, 256),     │    394,240 │ embedding_4[0][0] │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ [(None, 6, 256),  │    394,240 │ embedding_5[0][0… │
│                     │ (None, 256),      │            │ lstm_4[0][1],     │
│                     │ (None, 256)]      │            │ lstm_4[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 6, 17)     │      4,369 │ lstm_5[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 796,817 (3.04 MB)

 Trainable params: 796,817 (3.04 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - loss: 1.8896
Epoch 2/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 1.8794
Epoch 3/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 1.8688
Epoch 4/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 1.8573
Epoch 5/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 1.8442
Epoch 6/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 1.8287
Epoch 7/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 1.8098
Epoch 8/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 1.7864
Epoch 9/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 1.7570
Epoch 10/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 1.7194
Epoch 11/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - loss: 1.6718
Epoch 12/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: 1.6150
Epoch 13/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step - loss: 1.5679
Epoch 14/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step - loss: 1.5467
Epoch 15/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 1.5388
Epoch 16/300
1/1 ━━

In [9]:
# -----------------------------
# 11. Test Translation
# -----------------------------
print("\n--- Translation Results ---\n")

for sentence in input_texts:
    seq = input_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_input_len, padding='post')

    print("Input:", sentence)
    print("Output:", decode_sequence(seq))
    print("------")


--- Translation Results ---

Input: hello
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
Output: नमस्ते
------
Input: how are you
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Output: तिमीलाई कस्तो छ
------
Input: i am fine
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Output: म ठिक छु
------
Input: what is your name
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
Output: तिम्रो नाम के हो
------
Input: i love nepal
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━